# Continued Pretraining with Unsloth - Teaching New Knowledge

This notebook demonstrates continued pretraining using Unsloth. Continued pretraining is different from fine-tuning - it teaches the model new knowledge, vocabulary, or languages by training on raw text data.

## What we'll cover:
- Installing Unsloth and dependencies
- Loading a model for continued pretraining
- Using raw text data (not instruction-response pairs)
- Training with next-token prediction objective
- Testing the model's new knowledge

## About Continued Pretraining:
Continued pretraining is used to:
- Teach models new languages
- Add domain-specific knowledge (medical, legal, scientific)
- Update model knowledge with recent information
- Adapt models to specific writing styles or formats

## Key Differences from Fine-tuning:
- **Fine-tuning**: Uses instruction-response pairs, teaches the model to follow instructions
- **Continued Pretraining**: Uses raw text, teaches the model new knowledge/vocabulary/language

## Dataset Format:
Continued pretraining requires:
- Plain text data (no instruction-response structure)
- Large amounts of text in the target domain/language
- Simple format: just a "text" column with raw content

## Use Cases:
- Teaching a model a new language (e.g., making an English model understand French)
- Adding specialized knowledge (medical terminology, legal concepts)
- Updating knowledge cutoff with recent information
- Domain adaptation (general model to coding-specific model)

In [1]:
# Install Unsloth and dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kit3vxql/unsloth_98f8f0464e4d45deb4142954da66a17d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kit3vxql/unsloth_98f8f0464e4d45deb4142954da66a17d
  Resolved https://github.com/unslothai/unsloth.git to commit c69dbbf2994f46ca94c0d2d42a5f04a23bb9525e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 133.9 MB/s eta 0:00:00

## Import Required Libraries

Import the necessary libraries for continued pretraining. We'll use:
- `FastLanguageModel` from Unsloth for model loading
- `SFTTrainer` from TRL (works for both fine-tuning and continued pretraining)
- `TrainingArguments` for training configuration
- Standard PyTorch and datasets libraries

In [2]:
# Import necessary libraries
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer
from datasets import load_dataset

# Check if GPU is available
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Available: True
GPU Name: NVIDIA A100-SXM4-40GB
GPU Memory: 42.47 GB


## Model Configuration for Continued Pretraining

For continued pretraining, we'll use LoRA with 4-bit quantization for memory efficiency.

Configuration parameters:
- `max_seq_length`: Maximum sequence length (2048 tokens)
- `dtype`: Data type (None for auto-detection)
- `load_in_4bit`: True for memory-efficient 4-bit quantization
- `model_name`: We'll use SmolLM2 135M for consistency

Continued pretraining can be done with:
- Full fine-tuning (all parameters updated)
- LoRA (more memory efficient)

We'll use LoRA for efficiency, but the training objective is different - we're training on raw text to learn new knowledge rather than instruction-following.

In [3]:
# Configuration parameters for continued pretraining
max_seq_length = 2048  # Maximum sequence length
dtype = None  # Auto-detect dtype
load_in_4bit = True  # Use 4-bit quantization for efficiency

# Model selection - using SmolLM2 135M
model_name = "unsloth/SmolLM2-135M-Instruct"

print(f"Model: {model_name}")
print(f"Max Sequence Length: {max_seq_length}")
print(f"4-bit Quantization: {load_in_4bit}")
print("Continued pretraining mode - teaching new knowledge")

Model: unsloth/SmolLM2-135M-Instruct
Max Sequence Length: 2048
4-bit Quantization: True
Continued pretraining mode - teaching new knowledge


## Load Model and Configure LoRA Adapters

We load the model with 4-bit quantization and add LoRA adapters for continued pretraining.

For continued pretraining:
- We use the same LoRA setup as fine-tuning
- The difference is in the training data (raw text) and objective (next-token prediction)
- The model learns new vocabulary, knowledge, or language patterns
- Training is typically done for longer than fine-tuning

The LoRA configuration allows efficient training while teaching the model new capabilities.

In [4]:
# Load model and tokenizer with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Add LoRA adapters for continued pretraining
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("Model loaded successfully for continued pretraining!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
trainable_percentage = 100 * sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters())
print(f"Trainable percentage: {trainable_percentage:.2f}%")

==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2025.11.3 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


Model loaded successfully for continued pretraining!
Total parameters: 86,315,904
Trainable parameters: 4,884,480
Trainable percentage: 5.66%


## Load Dataset for Continued Pretraining

For continued pretraining, we need raw text data in a specific domain or language.

Examples of suitable datasets:
- **Wikipedia in different languages**: For teaching new languages
- **ArXiv papers**: For scientific knowledge
- **Code repositories**: For programming knowledge
- **Books/articles in specific domains**: For d

In [5]:
# Load a continued pretraining dataset
# We'll use a subset of Wikipedia or a simple text dataset
# For demo purposes, we'll use the "wikitext" dataset (English)
# You could replace this with non-English Wikipedia for language learning

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# Take a smaller subset for quick demo
dataset = dataset.select(range(min(1000, len(dataset))))

print(f"Dataset loaded successfully!")
print(f"Number of examples: {len(dataset)}")
print(f"Dataset columns: {dataset.column_names}")
print("\nFirst example (first 500 characters):")
print(dataset[0]['text'][:500])

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Dataset loaded successfully!
Number of examples: 1000
Dataset columns: ['text']

First example (first 500 characters):



## Format Dataset for Continued Pretraining

For continued pretraining, we need to:
- Ensure the dataset has a "text" column with raw text
- Filter out empty or very short texts
- Optionally add special tokens (EOS token at the end of each text)

The model will learn by predicting the next token in the sequence, which teaches it:
- Language patterns and grammar
- Vocabulary and word usage
- Domain-specific knowledge
- Writing style and structure

This is the same objective used in the original pretraining, but now on specialized data.

In [7]:
# Format dataset for continued pretraining
def format_pretraining_dataset(examples):
    """
    Format dataset for continued pretraining.
    Add EOS token to mark end of sequences.
    Filter out empty texts.
    """
    EOS_TOKEN = tokenizer.eos_token
    texts = []

    for text in examples['text']:
        # Skip empty or very short texts
        if text and len(text.strip()) > 10:
            # Add EOS token at the end
            formatted_text = text.strip() + EOS_TOKEN
            texts.append(formatted_text)

    return {'text': texts}

# Apply formatting and filter
formatted_dataset = dataset.map(
    format_pretraining_dataset,
    batched=True,
    remove_columns=dataset.column_names
)

# Remove any empty entries
formatted_dataset = formatted_dataset.filter(lambda x: len(x['text']) > 0)

print("Dataset formatted for continued pretraining!")
print(f"Number of examples after filtering: {len(formatted_dataset)}")
print("\nFormatted example (first 500 characters):")
print(formatted_dataset[0]['text'][:500])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/644 [00:00<?, ? examples/s]

Dataset formatted for continued pretraining!
Number of examples after filtering: 644

Formatted example (first 500 characters):
= Valkyria Chronicles III =<|im_end|>


## Setup Training Configuration for Continued Pretraining

Continued pretraining typically uses:
- Similar hyperparameters to fine-tuning
- Often trained for more steps/epochs (learning new knowledge takes time)
- Lower learning rate sometimes used for stability
- Language modeling objective (next-token prediction)

Key parameters:
- `per_device_train_batch_size`: Batch size per device
- `gradient_accumulation_steps`: Gradient accumulation
- `learning_rate`: Learning rate (sometimes lower for continued pretraining)
- `num_train_epochs`: Often set to multiple epochs for continued pretraining
- `max_steps`: Or use max_steps for controlled training

For this demo, we'll use a short training run to demonstrate the process.

In [8]:
# Setup training arguments for continued pretraining
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=100,  # More steps than fine-tuning typically
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs_continued_pretraining",
    report_to="none",
)

print("Training arguments configured for continued pretraining!")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total training steps: {training_args.max_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print("\nThe model will learn from raw text using next-token prediction")

Training arguments configured for continued pretraining!
Effective batch size: 8
Total training steps: 100
Learning rate: 0.0002

The model will learn from raw text using next-token prediction


## Initialize the Trainer for Continued Pretraining

For continued pretraining, we use `SFTTrainer` but with important differences:
- We're training on raw text, not instruction-response pairs
- The model learns by predicting the next token in sequences
- This teaches new vocabulary, knowledge, and patterns

Key parameters:
- `model`: The model with LoRA adapters
- `tokenizer`: For processing text
- `train_dataset`: Raw text dataset
- `dataset_text_field`: Column containing the text ("text")
- `max_seq_length`: Maximum sequence length
- `packing`: Can be True to pack multiple short texts into one sequence (more efficient)

The training objective is language modeling - predicting what comes next in the text.

In [9]:
# Initialize the SFTTrainer for continued pretraining
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Set to True for more efficient training with short texts
    args=training_args,
)

print("Trainer initialized for continued pretraining!")
print(f"Training dataset size: {len(trainer.train_dataset)}")
print(f"Training mode: Language modeling (next-token prediction)")
print(f"The model will learn patterns, vocabulary, and knowledge from raw text")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/644 [00:00<?, ? examples/s]

Trainer initialized for continued pretraining!
Training dataset size: 644
Training mode: Language modeling (next-token prediction)
The model will learn patterns, vocabulary, and knowledge from raw text


## Train the Model with Continued Pretraining

Now we start the continued pretraining process. The trainer will:
- Process raw text data
- Train the model to predict the next token in sequences
- Update model parameters to learn new patterns, vocabulary, and knowledge
- Log training metrics (loss should decrease as model learns)

Expected behavior:
- Training loss decreases as the model learns the text patterns
- The model absorbs new vocabulary and knowledge
- Takes longer than fine-tuning (learning new knowledge is harder than learning to follow instructions)
- Multiple epochs often needed for good results

For production continued pretraining:
- Train for many more steps (thousands to millions)
- Use larger datasets
- Consider using multiple epochs
- Monitor perplexity to evaluate learning

In [10]:
# Start continued pretraining
print("Starting continued pretraining...")
print("The model is learning new knowledge from raw text...")
trainer_stats = trainer.train()

print("\nContinued pretraining completed!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training samples per second: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print("\nThe model has learned new patterns and knowledge from the training data!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting continued pretraining...
The model is learning new knowledge from raw text...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 644 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,884,480 of 139,400,064 (3.50% trained)


Step,Training Loss
1,4.211000
2,3.776900
3,4.457300
4,3.999400
5,3.967000
6,4.592600
7,3.817000
8,3.798800
9,3.672000
10,3.805600



Continued pretraining completed!
Training loss: 3.6679
Training time: 152.50 seconds
Training samples per second: 5.25

The model has learned new patterns and knowledge from the training data!


## Test the Continued Pretrained Model and Save

After continued pretraining, let's test if the model has learned from the new data and then save it.

We'll:
1. Test the model with text generation to see if it uses patterns from the training data
2. Save the model for future use

The model should now have knowledge/patterns from the continued pretraining data incorporated into its responses.

In [11]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Test the model with a prompt related to the training data
test_prompt = """The history of """

# Tokenize the input
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# Generate response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("Testing continued pretrained model:")
print("=" * 50)
print(f"Prompt: {test_prompt}")
print("Completion: ", end="")
outputs = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=100,
    use_cache=True,
    temperature=0.7,
    top_p=0.9
)
print("=" * 50)

# Save the continued pretrained model
print("\nSaving the continued pretrained model...")
model_save_path = "smollm2_135m_continued_pretrained"

# Save LoRA adapters
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"\nLoRA adapters saved to: {model_save_path}")

# Optionally save merged model
merged_path = "smollm2_135m_continued_pretrained_merged"
model.save_pretrained_merged(
    merged_path,
    tokenizer,
    save_method="merged_16bit",
)

print(f"Merged model saved to: {merged_path}")

print("\n" + "=" * 50)
print("CONTINUED PRETRAINING COMPLETED")
print("=" * 50)
print("Summary of all 5 Colabs:")
print("1. Full Fine-tuning - Updated all model parameters")
print("2. LoRA Fine-tuning - Parameter-efficient with adapters")
print("3. DPO - Aligned model with human preferences")
print("4. GRPO - Trained reasoning capabilities")
print("5. Continued Pretraining - Taught new knowledge/language")
print("=" * 50)
print("\nAll training methods completed successfully!")

Testing continued pretrained model:
Prompt: The history of 
Completion: 19th-century American women is marked by a range of significant events and women's rights movements. The first women's rights movement began in 1848 with the Seneca Falls Convention in Seneca Falls, New York, where women were granted the right to vote in 1848. The first women's rights convention in 1860 was the 1860 Women's Rights Convention in Seneca Falls, New York, which was the first women's rights convention in

Saving the continued pretrained model...

LoRA adapters saved to: smollm2_135m_continued_pretrained
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `smollm2_135m_continued_pretrained_merged`: 100%|██████████| 1/1 [00:00<00:00,  7.39it/s]


Successfully copied all 1 files from cache to `smollm2_135m_continued_pretrained_merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]


Unsloth: Merge process complete. Saved to `/content/smollm2_135m_continued_pretrained_merged`
Merged model saved to: smollm2_135m_continued_pretrained_merged

CONTINUED PRETRAINING COMPLETED
Summary of all 5 Colabs:
1. Full Fine-tuning - Updated all model parameters
2. LoRA Fine-tuning - Parameter-efficient with adapters
3. DPO - Aligned model with human preferences
4. GRPO - Trained reasoning capabilities
5. Continued Pretraining - Taught new knowledge/language

All training methods completed successfully!
